In [6]:
#Getting environment variables from .env file
from dotenv import load_dotenv
load_dotenv()

True

In [7]:
import os
import google.generativeai as genai
# Set up Gemini API client
genai.configure(api_key=os.getenv("GEMINI_API_KEY"))
model = genai.GenerativeModel('gemini-3.1-flash-lite')


c:\Users\acer\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\acer\AppData\Local\Temp\ipykernel_5248\349812897.py:2: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [31]:
# Helper Function 1: Add a user message 
def add_user_message(history, user_input):
    history.append({
        'role': 'user',
        'parts': [user_input]  
    })

# Helper Function 2: Add an assistant message
def add_assistant_message(history, assistant_response):
    history.append({
        'role': 'assistant',
        'parts': [assistant_response]  
    })

# Helper Function 3: To get message from the model
def chat(history,stop_sequences=None,max_output_tokens=2048,temperature=0.7):
    generation_options = {
        'stop_sequences': stop_sequences,
        'max_output_tokens': max_output_tokens,
        'temperature': temperature,
    }
    response = model.generate_content(
        history,
        generation_config=generation_options,
        stream=True
    )

    full_response = ""
    print("Assistant: ", end="", flush=True)
    for chunk in response:
        full_response += chunk.text
        print(chunk.text, end="", flush=True)
    print() # Adds a clean closing line breaks after streaming finishes
    return full_response
    

In [32]:
# Creating a dataset generation function
def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
  {
    "task": "Description of task",
  },
  ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)


In [33]:
# Testing the dataset generation function
import json
dataset = generate_dataset()
print(dataset)

Assistant: 
[
  {
    "task": "Write a Python function that uses Boto3 to list all S3 bucket names in the current AWS account."
  },
  {
    "task": "Write a JSON IAM policy document that allows the 's3:GetObject' action on all objects within a specific bucket named 'my-app-assets'."
  },
  {
    "task": "Write a Regex pattern that matches a standard AWS IAM Role ARN format (e.g., arn:aws:iam::123456789012:role/service-role/role-name)."
  }
]

[{'task': 'Write a Python function that uses Boto3 to list all S3 bucket names in the current AWS account.'}, {'task': "Write a JSON IAM policy document that allows the 's3:GetObject' action on all objects within a specific bucket named 'my-app-assets'."}, {'task': 'Write a Regex pattern that matches a standard AWS IAM Role ARN format (e.g., arn:aws:iam::123456789012:role/service-role/role-name).'}]


In [11]:
# Saving the dataset to a JSON file
with open('dataset.json', 'w') as f:
    json.dump(dataset, f, indent=2)

In [34]:
 # Created a Model Grader
def grade_by_model(test_case, output):
    # Extract the string values from your dataset dictionaries cleanly
    task_text = test_case["task"]
    
    # Corrected f-string mappings to read the true input variables
    eval_prompt = f"""
You are an expert code reviewer. Evaluate this AI-generated solution.

Task: {task_text}
Solution: {output}

Provide your evaluation as a structured JSON object with:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your assessment
- "score": A number between 1-10
"""

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    
    # Call your chat executor with the correct stop sequences string array
    eval_text = chat(messages, stop_sequences=["```"])
    
    # Clean up the markdown wrappers so json.loads extracts cleanly
    clean_json = eval_text.replace("```json", "").replace("```", "").strip()
    return json.loads(clean_json)

In [35]:
# Added a run prompt function to take teh test case and merge with prompt template
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [40]:
import re
import ast
import json

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0

# FIXED: Fallback to scanning text strings if the explicit 'format' key is missing!
def grade_syntax(response, test_case):
    # Use .get() with a default value of None so it never throws a KeyError
    fmt = test_case.get("format", None)
    
    # If the format key isn't in your JSON file, let's auto-detect it using keywords
    if fmt is None:
        task_text = test_case.get("task", "").lower()
        if "json" in task_text:
            fmt = "json"
        elif "regex" in task_text or "regular expression" in task_text:
            fmt = "regex"
        else:
            fmt = "python" # Default fallback for code scripts
            
    # Run validators based on the resolved format type
    if fmt == "json":
        return validate_json(response)
    elif fmt == "python":
        return validate_python(response)
    elif fmt == "regex":
        return validate_regex(response)
    else:
        return 0

In [42]:
# A test case function to run the prompt on a single test case and give it the score 
def run_test_case(test_case):
    output = run_prompt(test_case)
    
    # Grade the output
    model_grade = grade_by_model(test_case, output)
    model_score= model_grade["score"]
    reasoning = model_grade["reasoning"]
    code_score = grade_syntax(output, test_case)
    score = (model_score + code_score) / 2  # Simple average of model grade and syntax validation
    return {
        "output": output, 
        "test_case": test_case, 
        "score": score,
        "reasoning": reasoning
    }

In [43]:
# A run eval function to run the test case function on all test cases in the dataset and return results
from statistics import mean

def run_eval(dataset):
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")
    
    return results

In [44]:
# Running the evaluation
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)
print(json.dumps(results, indent=2))

Assistant: To list all S3 bucket names in your AWS account using Boto3, you need to use the `list_buckets` method provided by the S3 client.

### Prerequisites
1. **Install Boto3:** `pip install boto3`
2. **Configure Credentials:** Ensure your AWS credentials (AWS Access Key ID, Secret Access Key) are configured via the AWS CLI (`aws configure`), environment variables, or an IAM role.

### The Python Code

```python
import boto3

def list_s3_buckets():
    """
    Lists all S3 bucket names in the current AWS account.
    """
    # Create an S3 client
    s3_client = boto3.client('s3')

    try:
        # Call S3 to list buckets
        response = s3_client.list_buckets()

        # Extract bucket names from the response
        bucket_names = [bucket['Name'] for bucket in response['Buckets']]
        
        return bucket_names

    except Exception as e:
        print(f"Error occurred: {e}")
        return []

# Example usage:
if __name__ == "__main__":
    buckets = list_s3_buckets(

In [45]:
print(json.dumps(results, indent=2))

[
  {
    "output": "To list all S3 bucket names in your AWS account using Boto3, you need to use the `list_buckets` method provided by the S3 client.\n\n### Prerequisites\n1. **Install Boto3:** `pip install boto3`\n2. **Configure Credentials:** Ensure your AWS credentials (AWS Access Key ID, Secret Access Key) are configured via the AWS CLI (`aws configure`), environment variables, or an IAM role.\n\n### The Python Code\n\n```python\nimport boto3\n\ndef list_s3_buckets():\n    \"\"\"\n    Lists all S3 bucket names in the current AWS account.\n    \"\"\"\n    # Create an S3 client\n    s3_client = boto3.client('s3')\n\n    try:\n        # Call S3 to list buckets\n        response = s3_client.list_buckets()\n\n        # Extract bucket names from the response\n        bucket_names = [bucket['Name'] for bucket in response['Buckets']]\n        \n        return bucket_names\n\n    except Exception as e:\n        print(f\"Error occurred: {e}\")\n        return []\n\n# Example usage:\nif __na